Moneyball: Prediction the amount of wins for a baseball team that season based on next generation statistics.

In [ ]:
!pip install pandas
!pip install numpy



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Data Importing and Pre-processing (100 Points)
Import dataset and describe characteristics such as dimensions, data types, file types, and import methods used
Clean, wrangle, and handle missing data, duplicate data, etc.
Encode any categorical variables
Perform feature engineering on the dataset
Transform data appropriately using techniques such as aggregation, normalization, and feature construction
Reduce redundant data and perform need based discretization

In [27]:
import numpy as np
import pandas as pd

# Import dataset
df = pd.read_csv('baseball.csv')

#describe characteristics such as dimensions, data types, file types
print(f'Shape: {df.shape}')
print(f'Dtypes: {df.dtypes}')
print("\nColumns:\n", df.columns.tolist())
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicates:", df.duplicated().sum())

#drop duplicates (should be none)
df = df.drop_duplicates().copy()

#because teams relocated/changed named, map the names from the same franchise but different team names
franchise_map = {
    "CAL": "LAA", "ANA": "LAA", "LAA": "LAA",
    "FLA": "MIA", "MIA": "MIA",
    "MON": "WSN", "WSN": "WSN",
    "TBD": "TBR", "TBR": "TBR",
    "MLN": "ATL", "ATL": "ATL",
    "WSA": "TEX", "TEX": "TEX",
    "KCA": "OAK", "OAK": "OAK",
    "SEP": "MIL", "MIL": "MIL"
}

df["Franchise"] = df["Team"].replace(franchise_map)
df = df.drop(columns=["Team"])

#drop opponent values because they are absent before 1999, RankSeason and RankPlayoffs because they are only present for playoff teams, and Playoffs because
#it is downstream from wins and causes data leakage
drop_cols = ['OOBP', 'OSLG', 'RankSeason', 'RankPlayoffs', 'Playoffs']
df = df.drop(columns=drop_cols)

#feature engineering
df = df.sort_values(['Franchise', 'Year']).copy()
grouped = df.groupby('Franchise')

df["run_diff"] = df["RS"] - df["RA"]
df["RS_per_game"] = df["RS"] / df["G"]
df["RA_per_game"] = df["RA"] / df["G"]
df["OPS"] = df["OBP"] + df["SLG"]

df["prev_wins"] = grouped["W"].shift(1)
df["wins_3yr_avg"] = grouped["W"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df["prev_run_diff"] = grouped["run_diff"].shift(1)
df["OBP_3yr_avg"] = grouped["OBP"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df["SLG_3yr_avg"] = grouped["SLG"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

#fill missing values caused by rolling/lagging
df["prev_wins"] = df["prev_wins"].fillna(df["wins_3yr_avg"])
df["wins_3yr_avg"] = df["wins_3yr_avg"].fillna(df["wins_3yr_avg"].median())
df["prev_run_diff"] = df["prev_run_diff"].fillna(0)
df["OBP_3yr_avg"] = df["OBP_3yr_avg"].fillna(df["OBP_3yr_avg"].median())
df["SLG_3yr_avg"] = df["SLG_3yr_avg"].fillna(df["SLG_3yr_avg"].median())

#discretization
df["era"] = pd.cut(
    df["Year"],
    bins=[1961, 1976, 1993, 1998, 2012],
    labels=["1962_1976", "1977_1993", "1994_1998", "1999_2012"]
)

#encode
df = pd.get_dummies(df, columns=['League', 'Franchise', 'era'], drop_first=True)

#csv for easier viewing
df = df.sort_values(['Year'], ascending=False).reset_index(drop=True)
df.to_csv("baseball_cleaned.csv", index=False)

#we will normalize after split


Shape: (1232, 15)
Dtypes: Team                str
League              str
Year              int64
RS                int64
RA                int64
W                 int64
OBP             float64
SLG             float64
BA              float64
Playoffs          int64
RankSeason      float64
RankPlayoffs    float64
G                 int64
OOBP            float64
OSLG            float64
dtype: object

Columns:
 ['Team', 'League', 'Year', 'RS', 'RA', 'W', 'OBP', 'SLG', 'BA', 'Playoffs', 'RankSeason', 'RankPlayoffs', 'G', 'OOBP', 'OSLG']

Missing values:
 Team              0
League            0
Year              0
RS                0
RA                0
W                 0
OBP               0
SLG               0
BA                0
Playoffs          0
RankSeason      988
RankPlayoffs    988
G                 0
OOBP            812
OSLG            812
dtype: int64

Duplicates: 0


Data Analysis and Visualization (100 Points)
Identify categorical, ordinal, and numerical variables within data
Provide measures of centrality and distribution with visualizations
Diagnose for correlations between variables and determine independent and dependent variables
Perform exploratory analysis in combination with visualization techniques to discover patterns and features of interest
Create visualizations that allow for the discovery of insights in the data

Data Analytics (100 Points)
Determine the need for a supervised or unsupervised learning method and identify dependent and independent variables
Choose and provide reasoning for the selected metric or metrics employed to assess your model.
Train, test, cross validate, and provide performance metrics for model results
Try multiple different types of algorithms to determine the best model for your dataset
Analyze your model performance